# Estimation of Lithium-Ion Battery SOH Based on a Hybrid Transformer-KAN Model

**Paper:** Chen, Z., Lu, J., Wei, Q., Wen, J., Wang, Y., Li, K., Xu, A. (2025). *Estimation of Lithium-Ion Battery SOH Based on a Hybrid Transformer-KAN Model.* Electronics, 14(24), 4859. DOI: 10.3390/electronics14244859.

**Carpeta origen:** `Papers/Ciencia, energía nuclear y química/Estimation_of_Lithium-Ion_Battery_SOH_Based_on_a_H.pdf`

## Como se usan las KAN en este paper

El paper propone **Transformer-KAN**, un modelo hibrido para estimar el State of Health (SOH) de baterias de ion-litio a partir de secuencias de indicadores de salud (Health Features, HF) extraidos de los ciclos de carga. La idea central es sustituir el decodificador estandar de un Transformer (una FFN dada por Ec. 7 con activacion ReLU fija) por una **capa KAN**, que reemplaza los pesos escalares de esa FFN por funciones B-spline aprendibles situadas en las aristas del grafo, aportando mayor capacidad de aproximacion no lineal e interpretabilidad intrinseca (la forma de cada funcion aprendida puede examinarse tras el entrenamiento).

**Extraccion de Health Features (Seccion 2).** SOH se define como el cociente entre la capacidad actual y la capacidad nominal:

$$\text{SOH} = \frac{C_{actual}}{C_{rated}} \times 100\%$$

A partir de las curvas de voltaje, corriente y temperatura durante la carga CC-CV, el paper extrae cinco indicadores (HF1-HF5) con correlacion de Pearson (Ec. 3) superior a 0.98 con el SOH en la mayoria de los casos: HF1 (tiempo de subida de voltaje entre 3.9V y 4.1V en fase CC), HF2 (area bajo la curva de voltaje entre 3.9V y 4.2V), HF3 (area bajo la curva de corriente entre 1.5A y 0.6A en fase CV), HF4 (instante del pico de temperatura durante la carga) y HF5 (pico de la curva de capacidad incremental $dQ/dV$, Ec. 2).

**Arquitectura hibrida (Seccion 3).** El encoder Transformer (Fig. 7) captura dependencias temporales de largo alcance en la secuencia de HFs mediante atencion multi-cabeza:

$$\text{Multihead}(Q,K,V)=\text{Concat}(head_1,\dots,head_n)W^O,\qquad head_i=\text{Attention}(QW_i^Q,KW_i^K,VW_i^W)$$
$$\text{Attention}(Q,K,V)=\text{SoftmaxFun}\!\left(\frac{QK^T}{\sqrt{d_n}}\right)V$$

La salida del encoder $H_{trans}\in\mathbb{R}^{L\times d_{model}}$ se reduce con *global average pooling* a un vector $h\in\mathbb{R}^{d_{model}}$ que alimenta la capa KAN final, la cual produce la estimacion escalar de SOH (Algoritmo 1):

$$H_{trans}=\text{TransformerEncoder}(X_{seq}),\qquad h=\text{GlobalAveragePooling}(H_{trans}),\qquad \hat y=\text{KAN}(h)$$

**Capa KAN (Seccion 3.2).** Se fundamenta en el teorema de representacion de Kolmogorov-Arnold (Ec. 8):

$$f(x_1,\dots,x_n)=\sum_{i=1}^{2n+1}\Phi_i\!\left(\sum_{j=1}^{n}\phi_{ij}(x_j)\right)$$

donde cada funcion univariante $\phi_{ij}$ se implementa como una combinacion lineal de B-splines (Ec. 9):

$$\phi_{ij}(x_j)=\sum_{m=0}^{k}c_{ijm}\,B_{m,k}(x_j)$$

cuyas funciones base se construyen con la recursion de Cox-de Boor (Ec. 10-11):

$$B_{i,0}(t)=\begin{cases}1,& t_i\le t< t_{i+1}\\0,&\text{en otro caso}\end{cases}\qquad B_{i,k}(t)=\frac{t-t_i}{t_{i+k}-t_i}B_{i,k-1}(t)+\frac{t_{i+k+1}-t}{t_{i+k+1}-t_{i+1}}B_{i+1,k-1}(t)$$

La Tabla 2 del paper fija el tamano de rejilla (grid size = 5) y el orden de spline (k = 3) de esta capa, ademas de un termino base con escala 1.0 (rama residual tipo SiLU, propia del diseno original de KAN, sumada a la rama spline). La perdida (Algoritmo 1) es el error cuadratico medio entre SOH real y estimado, optimizado con Adam (lr inicial $10^{-3}$, reducido por 0.5 tras 20 epocas sin mejora) y validado con un esquema de **validacion cruzada entre baterias** (leave-one-battery-out): se entrena con dos baterias del dataset NASA y se evalua en la tercera, nunca vista durante el entrenamiento.

Este cuaderno reproduce fielmente el mecanismo central del paper: la extraccion de las cinco Health Features desde curvas de carga simuladas con la misma logica de calculo (tiempos de cruce de voltaje, areas bajo curva, pico de temperatura, pico de $dQ/dV$), el analisis de correlacion de Pearson, la capa KAN con splines-B (grid size 5, orden 3, rama base) implementada desde cero en PyTorch, el encoder Transformer (2 capas, 4 cabezas, FFN de dimension 128, dropout 0.3, Tabla 2), el modelo hibrido Transformer-KAN completo, y la validacion cruzada entre baterias comparandolo contra los mismos tres baselines del paper (Transformer puro, KAN puro y CNN-LSTM), usando datos sinteticos que preservan la estructura fisica del problema (ver Seccion 1: el dataset real NASA no esta empaquetado en este entorno).

## Repositorio publico

El paper **no declara** ningun repositorio de codigo propio: su seccion *Data Availability Statement* indica unicamente que "the original contributions presented in this study are included in the article", sin enlace a GitHub ni a ningun otro repositorio. Se realizo una busqueda adicional en GitHub ("Transformer-KAN" battery SOH, autores Chen/Lu/Wei/Wen/Wang/Li/Xu, Guangxi University of Science and Technology) sin encontrar ninguna implementacion oficial asociada a este trabajo especifico.

El dataset que si usa el paper es publico: el **NASA Li-Ion Battery Aging Dataset** (referencia [26] del paper, baterias B0005/B0006/B0007), disponible en https://c3.nasa.gov/dashlink/resources/133/. No esta accesible para descarga automatica en este entorno, por lo que se simulan datos sinteticos con la misma estructura fisica (ver Seccion 1).

Como referencia del marco general de las KAN (Ec. 8-12), esta disponible localmente el repositorio oficial **KindXiaoming/pykan** (`Kolmogorov-Arnold Networks/codigo/pykan`, tambien instalable via `pip install pykan`). Sin embargo, pykan no se usa directamente aqui: por razones de coste computacional y control fino sobre `grid_size`/`spline_order` (Tabla 2), la capa KAN se implementa desde cero en PyTorch siguiendo el patron rapido de **Blealtan/efficient-kan** (https://github.com/Blealtan/efficient-kan), una reimplementacion eficiente y ampliamente adoptada de la formulacion B-spline de Ec. 8-12 que es funcionalmente equivalente a la capa `KANLayer` de pykan.

In [ ]:
# Instalacion de dependencias (ejecutar si no estan ya instaladas en el entorno)
%pip install -q torch numpy matplotlib pandas scipy

In [ ]:
import numpy as np
import pandas as pd
import torch
import torch.nn as nn
import torch.nn.functional as F
import matplotlib.pyplot as plt

torch.manual_seed(0)
np.random.seed(0)
device = torch.device('cuda' if torch.cuda.is_available() else 'cpu')
print('Device:', device)

## 1. Datos sinteticos: curvas de degradacion de SOH (estructura del dataset NASA B0005/B0006/B0007)

El paper usa el **NASA Li-Ion Battery Aging Dataset** (ref. [26]): tres celdas 18650 (B0005, B0006, B0007) ciclando carga CC-CV / descarga hasta que la capacidad cae de 2 Ah a 1.4 Ah, con curvas de SOH **no monotonas** (Fig. 2 del paper): la tendencia general decrece, pero aparecen picos de "regeneracion de capacidad" por relajacion electroquimica durante los periodos de reposo entre ciclos. Ese dataset no esta empaquetado en este entorno, asi que generamos **tres baterias sinteticas** (S05, S06, S07, analogas a B0005/6/7) con la misma estructura: una tendencia de desgaste decreciente (potencia de segundo orden en el numero de ciclo) mas un termino de regeneracion cuasi-periodico (subida rapida seguida de relajacion exponencial, con periodo e intensidad distintos por bateria) mas ruido gaussiano, igual que se aprecia en la Fig. 2 del paper.

In [ ]:
def simulate_soh(n_cycles=165, soh0=0.99, fade_rate=0.0022, fade_rate2=6e-6, seed=0):
    """Curva SOH(ciclo) con tendencia decreciente + regeneracion cuasi-periodica + ruido (cf. Fig. 2)."""
    rng = np.random.default_rng(seed)
    n = np.arange(1, n_cycles + 1)
    trend = soh0 - fade_rate * n - fade_rate2 * n ** 2
    period = rng.integers(18, 28)
    phase = rng.integers(0, period)
    saw = ((n + phase) % period) / period          # 0 -> 1 diente de sierra
    regen = 0.018 * np.exp(-3 * saw)                 # subida (relajacion) seguida de decaimiento
    noise = rng.normal(0, 0.003, n_cycles)
    soh = np.clip(trend + regen + noise, 0.40, 1.05)  # fraccion 0-1 (SOH = fraccion * 100%, Ec. 1)
    return soh


BATTERY_CFG = {
    'S05': dict(fade_rate=0.00195, fade_rate2=5e-6, seed=5, n_cycles=165),
    'S06': dict(fade_rate=0.00235, fade_rate2=7e-6, seed=6, n_cycles=165),
    'S07': dict(fade_rate=0.00165, fade_rate2=4e-6, seed=7, n_cycles=165),
}

soh_data = {name: simulate_soh(**cfg) for name, cfg in BATTERY_CFG.items()}

fig, ax = plt.subplots(figsize=(6, 4))
for name, soh in soh_data.items():
    ax.plot(soh * 100, label=name)
ax.set_xlabel('numero de ciclo'); ax.set_ylabel('SOH (%)')
ax.set_title('SOH sintetico (cf. Fig. 2 del paper: B0005/B0006/B0007)')
ax.legend(); plt.tight_layout(); plt.show()

for name, soh in soh_data.items():
    print(f'{name}: SOH inicial={soh[0]*100:.1f}%  SOH final={soh[-1]*100:.1f}%  n_ciclos={len(soh)}')

## 2. Curvas de carga simuladas y extraccion de Health Features HF1-HF5 (Seccion 2, Ec. 1-2)

Para cada ciclo simulamos las curvas de voltaje $V(t)$, corriente $I(t)$ y temperatura $T(t)$ durante la carga CC-CV (cf. Fig. 3-5 del paper), parametrizadas por el SOH del ciclo: cuanto mas envejecida esta la bateria, **antes** se alcanza el voltaje de corte de 4.2V (fase CC mas corta, Fig. 3), **mas lenta** es la caida de corriente en fase CV (Fig. 4), y **antes** ocurre el pico de temperatura (Fig. 5). A partir de estas curvas extraemos HF1-HF4 con exactamente la misma logica de calculo que el paper (tiempos de cruce de voltaje por interpolacion lineal, areas bajo curva por integracion trapezoidal, argmax de temperatura). Para HF5 (pico de la curva IC = $dQ/dV$, Ec. 2), en vez de derivar numericamente una curva de voltaje suave -lo que no reproduciria los picos de la curva IC real, que provienen de transiciones de fase del material del electrodo y requieren, segun el propio paper, "interpolacion lineal y suavizado/filtrado para suprimir el ruido"- modelamos directamente la curva IC como una mezcla de dos gaussianas (pico principal ~3.55V + hombro secundario ~3.78V, cf. Fig. 6) cuya amplitud decrece con el envejecimiento, que es la forma cualitativa que reporta la Fig. 6 del paper.

In [ ]:
def simulate_charge_cycle(soh_frac, t_max=8000.0, n_t=400, rng=None):
    """V(t), I(t), T(t) durante carga CC-CV de un ciclo, parametrizadas por SOH (cf. Fig. 3-5)."""
    t = np.linspace(0, t_max, n_t)

    # --- Fase CC: voltaje sube 3.55V -> 4.2V; la duracion CC decrece con el envejecimiento (Fig. 3) ---
    V_min, V_cut = 3.55, 4.2
    t_cc = 1500.0 + 2300.0 * soh_frac
    v_cc = V_min + (V_cut - V_min) * np.clip(t / t_cc, 0, 1) ** 0.55
    V = np.where(t <= t_cc, v_cc, V_cut)
    if rng is not None:
        V = V + rng.normal(0, 0.003, n_t)

    # --- Fase CV: corriente decae 1.5A -> ~0; la constante de tiempo crece con el envejecimiento (Fig. 4) ---
    I_max = 1.5
    tau_I = 850.0 + 1500.0 * (1 - soh_frac)
    I = np.where(t <= t_cc, I_max, I_max * np.exp(-(t - t_cc) / tau_I))
    if rng is not None:
        I = np.clip(I + rng.normal(0, 0.004, n_t), 0, I_max)

    # --- Temperatura: pico unico, cuyo instante se adelanta con el envejecimiento (Fig. 5) ---
    T_amb = 24.5
    t_peak = np.clip(900.0 + 2200.0 * soh_frac + (rng.normal(0, 60) if rng is not None else 0.0), 300, t_max - 300)
    sigma_l = 0.35 * t_peak + 250
    sigma_r = 0.30 * (t_max - t_peak) + 400
    amp = 5.0 + (rng.normal(0, 0.15) if rng is not None else 0.0)
    T = np.where(t <= t_peak,
                 T_amb - 0.5 + amp * np.exp(-0.5 * ((t - t_peak) / sigma_l) ** 2),
                 T_amb - 0.5 + amp * np.exp(-0.5 * ((t - t_peak) / sigma_r) ** 2))
    if rng is not None:
        T = T + rng.normal(0, 0.05, n_t)
    return t, V, I, T


def simulate_ic_curve(soh_frac, rng=None, n_v=300):
    """Curva IC = dQ/dV modelada como mezcla de 2 gaussianas cuya amplitud decrece con el envejecimiento (Fig. 6)."""
    V_grid = np.linspace(2.8, 4.0, n_v)
    peak1, peak2 = 3.55 + 0.02 * (1 - soh_frac), 3.78
    amp1 = 1.4 + 4.2 * soh_frac
    amp2 = 0.4 + 1.0 * soh_frac
    ic = amp1 * np.exp(-0.5 * ((V_grid - peak1) / 0.10) ** 2) + amp2 * np.exp(-0.5 * ((V_grid - peak2) / 0.12) ** 2)
    if rng is not None:
        ic = ic + rng.normal(0, 0.05, n_v)
    return V_grid, ic


def _cross_time(V_arr, t_arr, level):
    """Instante en que V_arr cruza 'level' (interpolacion lineal entre muestras)."""
    idx = np.where(V_arr >= level)[0]
    if len(idx) == 0:
        return t_arr[-1]
    i = idx[0]
    if i == 0:
        return t_arr[0]
    v0, v1, t0, t1 = V_arr[i - 1], V_arr[i], t_arr[i - 1], t_arr[i]
    return t1 if v1 == v0 else t0 + (level - v0) * (t1 - t0) / (v1 - v0)


def extract_health_features(t, V, I, T, soh_frac, rng):
    """HF1-HF5 con la misma logica que la Seccion 2 del paper."""
    t39, t41, t42 = _cross_time(V, t, 3.9), _cross_time(V, t, 4.1), _cross_time(V, t, 4.2)
    HF1 = t41 - t39                                                        # tiempo de subida 3.9V->4.1V (CC)

    mask2 = (t >= t39) & (t <= t42)
    HF2 = np.trapezoid(V[mask2], t[mask2]) if mask2.sum() > 1 else 0.0      # area bajo V, 3.9V-4.2V

    mask3 = (I <= 1.5) & (I >= 0.6) & (t >= t42)
    HF3 = np.trapezoid(I[mask3], t[mask3]) if mask3.sum() > 1 else 0.0      # area bajo I, 1.5A-0.6A (CV)

    HF4 = t[np.argmax(T)]                                                   # instante del pico de temperatura

    _, ic = simulate_ic_curve(soh_frac, rng=rng)
    HF5 = ic.max()                                                          # pico de la curva IC = dQ/dV

    return HF1, HF2, HF3, HF4, HF5


def build_battery_dataset(soh, seed):
    rng = np.random.default_rng(seed + 1000)
    HFs = [extract_health_features(*simulate_charge_cycle(s, rng=rng)[:4], s, rng) for s in soh]
    return np.array(HFs)


hf_data = {name: build_battery_dataset(soh_data[name], cfg['seed']) for name, cfg in BATTERY_CFG.items()}

# Ejemplo de curvas en distintos ciclos de S05 (cf. Fig. 3-6, bateria "saludable" vs. "envejecida")
rng_plot = np.random.default_rng(0)
fig, axes = plt.subplots(1, 3, figsize=(15, 4))
for c, color in zip([5, 60, 120, 160], ['tab:blue', 'tab:orange', 'tab:green', 'tab:red']):
    t, V, I, T = simulate_charge_cycle(soh_data['S05'][c], rng=rng_plot)
    axes[0].plot(t, V, color=color, label=f'ciclo {c}')
    axes[1].plot(t, I, color=color, label=f'ciclo {c}')
    axes[2].plot(t, T, color=color, label=f'ciclo {c}')
axes[0].set_title('Voltaje (cf. Fig. 3)'); axes[0].set_xlabel('t (s)'); axes[0].set_ylabel('V')
axes[1].set_title('Corriente (cf. Fig. 4)'); axes[1].set_xlabel('t (s)'); axes[1].set_ylabel('A')
axes[2].set_title('Temperatura (cf. Fig. 5)'); axes[2].set_xlabel('t (s)'); axes[2].set_ylabel('C')
for ax in axes: ax.legend(fontsize=8)
plt.tight_layout(); plt.show()

print('Forma de HF por bateria:', {k: v.shape for k, v in hf_data.items()})
print('\nEjemplo HF1-HF5 (S05, ciclos 5 y 160):')
print(pd.DataFrame(hf_data['S05'][[5, 160]], columns=[f'HF{i+1}' for i in range(5)], index=['ciclo 5', 'ciclo 160']))

## 3. Analisis de correlacion de Pearson entre Health Features y SOH (Ec. 3, Tabla 1)

$$r=\frac{\sum_{i=1}^n(x_i-\hat x)(y_i-\hat y)}{\sqrt{\sum_{i=1}^n(x_i-\hat x)^2\sum_{i=1}^n(y_i-\hat y)^2}}$$

El paper reporta $|r|>0.98$ para HF1, HF2, HF4, HF5 y una correlacion negativa moderada para HF3 (Tabla 1), justificando la seleccion del conjunto de 5 features. Calculamos la misma tabla sobre nuestras 3 baterias sinteticas.

In [ ]:
rows = {}
for name in BATTERY_CFG:
    rows[name] = [np.corrcoef(hf_data[name][:, i], soh_data[name])[0, 1] for i in range(5)]
corr_df = pd.DataFrame(rows, index=[f'HF{i+1}' for i in range(5)])

print('Correlacion de Pearson HFi vs SOH (analogo a la Tabla 1 del paper):')
corr_df.round(3)

## 4. Capa KAN: B-splines aprendibles sobre las aristas (Ec. 8-12, Tabla 2)

Implementamos `KANLinear` desde cero en PyTorch, siguiendo el patron eficiente de **Blealtan/efficient-kan** (funcionalmente equivalente a `KANLayer` de pykan): cada arista aplica una funcion univariante $\phi_{ij}(x_j)=\sum_m c_{ijm}B_{m,k}(x_j)$ (Ec. 9) mas una rama base residual tipo SiLU con escala 1.0 (Tabla 2: "Base function scale = 1.0", el diseno original de KAN). Las bases $B_{m,k}$ se evaluan con la recursion de Cox-de Boor (Ec. 10-11) de forma totalmente vectorizada. Usamos `grid_size=5` y `spline_order=3` exactamente como en la Tabla 2 del paper.

In [ ]:
class KANLinear(nn.Module):
    """Capa KAN (Ec. 8-12): phi_ij(x_j) = suma_m c_ijm * B_m,k(x_j)  +  rama base residual SiLU."""

    def __init__(self, in_features, out_features, grid_size=5, spline_order=3,
                 scale_base=1.0, scale_spline=1.0, grid_range=(0.0, 1.0)):
        super().__init__()
        self.in_features = in_features
        self.out_features = out_features
        self.grid_size = grid_size
        self.spline_order = spline_order
        self.scale_base = scale_base
        self.scale_spline = scale_spline

        h = (grid_range[1] - grid_range[0]) / grid_size
        grid = (torch.arange(-spline_order, grid_size + spline_order + 1, dtype=torch.float32) * h
                + grid_range[0])
        grid = grid.expand(in_features, -1).contiguous()
        self.register_buffer('grid', grid)   # (in_features, grid_size + 2*spline_order + 1)

        self.base_weight = nn.Parameter(torch.empty(out_features, in_features))
        self.spline_weight = nn.Parameter(torch.empty(out_features, in_features, grid_size + spline_order))
        self.base_activation = nn.SiLU()
        self.reset_parameters()

    def reset_parameters(self):
        nn.init.kaiming_uniform_(self.base_weight, a=5 ** 0.5)
        with torch.no_grad():
            noise = (torch.rand(self.grid_size + 1, self.in_features, self.out_features) - 0.5) * 0.05
            self.spline_weight.data.copy_(
                self.scale_spline * self._curve2coeff(self.grid.T[self.spline_order:-self.spline_order], noise)
            )

    def b_splines(self, x):
        """Recursion de Cox-de Boor (Ec. 10-11), vectorizada. x: (batch, in_features)."""
        grid = self.grid
        x = x.unsqueeze(-1)
        bases = ((x >= grid[:, :-1]) & (x < grid[:, 1:])).to(x.dtype)
        for k in range(1, self.spline_order + 1):
            left = (x - grid[:, :-(k + 1)]) / (grid[:, k:-1] - grid[:, :-(k + 1)]) * bases[:, :, :-1]
            right = (grid[:, k + 1:] - x) / (grid[:, k + 1:] - grid[:, 1:-k]) * bases[:, :, 1:]
            bases = left + right
        return bases  # (batch, in_features, grid_size + spline_order)

    def _curve2coeff(self, x, y):
        A = self.b_splines(x).transpose(0, 1)      # (in_features, n_puntos, coeff)
        B = y.transpose(0, 1)                       # (in_features, n_puntos, out_features)
        solution = torch.linalg.lstsq(A, B).solution  # (in_features, coeff, out_features)
        return solution.permute(2, 0, 1)             # (out_features, in_features, coeff)

    def forward(self, x):
        base_out = F.linear(self.base_activation(x), self.base_weight)         # rama residual (Tabla 2)
        b = self.b_splines(x).reshape(x.size(0), -1)
        spline_out = F.linear(b, self.spline_weight.reshape(self.out_features, -1))  # rama spline (Ec. 9)
        return self.scale_base * base_out + spline_out


class KAN(nn.Module):
    """Pila de capas KANLinear (Ec. 12: f(x1,x2) = Phi(suma_j phi_ij(x_j)))."""

    def __init__(self, layers_hidden, grid_size=5, spline_order=3, grid_range=(0.0, 1.0)):
        super().__init__()
        self.layers = nn.ModuleList([
            KANLinear(layers_hidden[i], layers_hidden[i + 1], grid_size=grid_size,
                      spline_order=spline_order, grid_range=grid_range)
            for i in range(len(layers_hidden) - 1)
        ])

    def forward(self, x):
        for layer in self.layers:
            x = layer(x)
        return x


# Prueba rapida: una capa KAN de 5 -> 8 -> 1 sobre un lote aleatorio
kan_test = KAN([5, 8, 1])
x_test = torch.rand(4, 5)
print('Salida KAN de prueba:', kan_test(x_test).shape)
print('Parametros de la capa KAN de prueba:', sum(p.numel() for p in kan_test.parameters()))

## 5. Encoder Transformer + capa KAN: el modelo hibrido Transformer-KAN (Fig. 7, Fig. 9, Algoritmo 1)

`TransformerKAN` implementa exactamente el flujo del Algoritmo 1: `X_seq -> TransformerEncoder -> GlobalAveragePooling -> KAN -> y_hat`. El encoder usa `nn.TransformerEncoderLayer` (atencion multi-cabeza Ec. 4-6 + FFN con ReLU Ec. 7 + conexiones residuales + LayerNorm, exactamente la Fig. 7 del paper) con los hiperparametros de la Tabla 2: **2 capas, 4 cabezas de atencion, FFN de dimension 128, dropout 0.3**. La FFN interna del encoder se deja como en el paper (Ec. 7, la parte "Transformer" de la arquitectura); es el **decodificador final** el que se sustituye por la capa `KAN` (Seccion 3.3: "we introduce a novel modification: replacing the standard FFN with a KAN layer").

Para la comparacion de la Seccion 7 (cf. Tabla 3) implementamos ademas los tres baselines del paper: **Transformer** puro (mismo encoder + cabeza MLP en vez de KAN), **KAN** puro (sin Transformer, aplicado directamente sobre la ventana temporal aplanada) y **CNN-LSTM** (Conv1D + LSTM, arquitectura estandar de referencia citada en la introduccion del paper, ref. [21]).

In [ ]:
L_WINDOW = 10  # longitud de la ventana temporal (numero de ciclos) que ve el Transformer


class TransformerKAN(nn.Module):
    """Modelo hibrido propuesto (Fig. 9, Algoritmo 1): encoder Transformer + capa KAN final."""

    def __init__(self, n_features=5, d_model=16, n_heads=4, n_layers=2, dim_ff=128, dropout=0.3, kan_hidden=12):
        super().__init__()
        self.input_proj = nn.Linear(n_features, d_model)
        enc_layer = nn.TransformerEncoderLayer(d_model=d_model, nhead=n_heads, dim_feedforward=dim_ff,
                                                dropout=dropout, activation='relu', batch_first=True)
        self.encoder = nn.TransformerEncoder(enc_layer, num_layers=n_layers)
        self.kan = KAN([d_model, kan_hidden, 1], grid_size=5, spline_order=3, grid_range=(-2.0, 2.0))

    def forward(self, x):                      # x: (batch, L, n_features)
        h = self.input_proj(x)
        h = self.encoder(h)                     # H_trans (Ec. 4-7)
        h = h.mean(dim=1)                        # GlobalAveragePooling
        return self.kan(h).squeeze(-1)           # KAN(h) -> SOH estimado (Ec. 8-12)


class TransformerMLP(nn.Module):
    """Baseline: Transformer estandar, con cabeza MLP densa en vez de KAN."""

    def __init__(self, n_features=5, d_model=16, n_heads=4, n_layers=2, dim_ff=128, dropout=0.3):
        super().__init__()
        self.input_proj = nn.Linear(n_features, d_model)
        enc_layer = nn.TransformerEncoderLayer(d_model=d_model, nhead=n_heads, dim_feedforward=dim_ff,
                                                dropout=dropout, activation='relu', batch_first=True)
        self.encoder = nn.TransformerEncoder(enc_layer, num_layers=n_layers)
        self.head = nn.Sequential(nn.Linear(d_model, 12), nn.ReLU(), nn.Linear(12, 1))

    def forward(self, x):
        h = self.input_proj(x)
        h = self.encoder(h).mean(dim=1)
        return self.head(h).squeeze(-1)


class KANOnly(nn.Module):
    """Baseline: KAN puro sobre la ventana temporal aplanada (sin Transformer)."""

    def __init__(self, L, n_features=5, hidden=16):
        super().__init__()
        self.kan = KAN([L * n_features, hidden, 1], grid_size=5, spline_order=3, grid_range=(-2.0, 2.0))

    def forward(self, x):
        b, L, f = x.shape
        return self.kan(x.reshape(b, L * f)).squeeze(-1)


class CNNLSTM(nn.Module):
    """Baseline CNN-LSTM (ref. [21] del paper)."""

    def __init__(self, n_features=5, cnn_ch=16, lstm_hidden=24):
        super().__init__()
        self.conv = nn.Conv1d(n_features, cnn_ch, kernel_size=3, padding=1)
        self.lstm = nn.LSTM(cnn_ch, lstm_hidden, batch_first=True)
        self.head = nn.Linear(lstm_hidden, 1)

    def forward(self, x):
        h = F.relu(self.conv(x.transpose(1, 2))).transpose(1, 2)
        out, _ = self.lstm(h)
        return self.head(out[:, -1]).squeeze(-1)


# Prueba de forma con un lote aleatorio (batch=4, L=10, 5 features)
x_smoke = torch.rand(4, L_WINDOW, 5)
for M in [TransformerKAN(), TransformerMLP(), KANOnly(L_WINDOW), CNNLSTM()]:
    y_smoke = M(x_smoke)
    n_params = sum(p.numel() for p in M.parameters())
    print(f'{type(M).__name__:16s} salida={tuple(y_smoke.shape)}  parametros={n_params}')

## 6. Secuencias, normalizacion min-max y entrenamiento (Seccion 3.4, Algoritmo 1)

Cada muestra de entrada $X_{seq}\in\mathbb{R}^{L\times 5}$ es una ventana deslizante de `L_WINDOW` ciclos consecutivos de HF1-HF5; la etiqueta es el SOH del ultimo ciclo de la ventana. Siguiendo la Seccion 3.4, normalizamos las features con min-max al rango $[0,1]$ (ajustado **solo** sobre las baterias de entrenamiento, para no filtrar informacion de la bateria de test). El entrenamiento usa Adam (lr inicial $10^{-3}$), `ReduceLROnPlateau` (factor 0.5) y perdida MSE (Algoritmo 1, linea 11), con tamano de lote 16 (Tabla 2).

In [ ]:
def make_sequences(hf, soh, L=L_WINDOW):
    X, y = [], []
    for i in range(L - 1, len(soh)):
        X.append(hf[i - L + 1:i + 1])
        y.append(soh[i])
    return np.array(X, dtype=np.float32), np.array(y, dtype=np.float32)


def fit_minmax(hf_list):
    all_hf = np.concatenate(hf_list, axis=0)
    return all_hf.min(axis=0), all_hf.max(axis=0)


def apply_minmax(hf, mn, mx):
    return (hf - mn) / (mx - mn + 1e-8)


def train_model(model, X_tr, y_tr, epochs=150, lr=1e-3, batch_size=16, patience=15):
    model.to(device)
    opt = torch.optim.Adam(model.parameters(), lr=lr)
    sched = torch.optim.lr_scheduler.ReduceLROnPlateau(opt, factor=0.5, patience=patience)
    Xt = torch.tensor(X_tr, dtype=torch.float32, device=device)
    yt = torch.tensor(y_tr, dtype=torch.float32, device=device)
    n = Xt.shape[0]
    history = []
    for epoch in range(epochs):
        model.train()
        perm = torch.randperm(n)
        epoch_loss = 0.0
        for i in range(0, n, batch_size):
            idx = perm[i:i + batch_size]
            opt.zero_grad()
            pred = model(Xt[idx])
            loss = F.mse_loss(pred, yt[idx])
            loss.backward()
            opt.step()
            epoch_loss += loss.item() * len(idx)
        epoch_loss /= n
        if not np.isfinite(epoch_loss):
            raise RuntimeError('Perdida no finita (NaN/Inf) durante el entrenamiento')
        sched.step(epoch_loss)
        history.append(epoch_loss)
    return history


def evaluate_model(model, X, y):
    model.eval()
    with torch.no_grad():
        pred = model(torch.tensor(X, dtype=torch.float32, device=device)).cpu().numpy()
    mae = float(np.mean(np.abs(pred - y)) * 100)    # en % de SOH (Ec. 13)
    rmse = float(np.sqrt(np.mean((pred - y) ** 2)) * 100)  # en % de SOH (Ec. 14)
    return pred, mae, rmse


print('Utilidades de entrenamiento y evaluacion definidas. L_WINDOW =', L_WINDOW)

## 7. Validacion cruzada entre baterias (leave-one-battery-out) y comparacion frente a los baselines (Tabla 3)

Reproducimos el protocolo de la Seccion 4.2: para cada bateria, se entrena con las **otras dos** y se evalua en la bateria restante, **nunca vista en entrenamiento** — la prueba de generalizacion mas exigente del paper. Repetimos esto para los 4 modelos (Transformer-KAN y los 3 baselines) y las 3 baterias sinteticas, midiendo RMSE y MAE (Ec. 13-14) en cada configuracion, igual que la Tabla 3 del paper.

In [ ]:
MODEL_BUILDERS = {
    'Transformer-KAN': lambda: TransformerKAN(),
    'Transformer': lambda: TransformerMLP(),
    'KAN': lambda: KANOnly(L_WINDOW),
    'CNN-LSTM': lambda: CNNLSTM(),
}

EPOCHS = 150
names = list(BATTERY_CFG.keys())
results_rows = []
preds_store = {}

for test_name in names:
    train_names = [n for n in names if n != test_name]
    mn, mx = fit_minmax([hf_data[n] for n in train_names])

    X_tr_list, y_tr_list = [], []
    for n in train_names:
        Xn, yn = make_sequences(apply_minmax(hf_data[n], mn, mx), soh_data[n])
        X_tr_list.append(Xn); y_tr_list.append(yn)
    X_tr, y_tr = np.concatenate(X_tr_list), np.concatenate(y_tr_list)
    X_te, y_te = make_sequences(apply_minmax(hf_data[test_name], mn, mx), soh_data[test_name])

    preds_store[test_name] = {'true': y_te}
    for model_name, builder in MODEL_BUILDERS.items():
        torch.manual_seed(0)
        model = builder()
        train_model(model, X_tr, y_tr, epochs=EPOCHS)
        pred, mae, rmse = evaluate_model(model, X_te, y_te)
        preds_store[test_name][model_name] = pred
        results_rows.append(dict(test_battery=test_name, model=model_name, RMSE=rmse, MAE=mae))
        print(f'test={test_name}  modelo={model_name:16s}  RMSE={rmse:.3f}%  MAE={mae:.3f}%')

results_df = pd.DataFrame(results_rows)
tabla_tipo_paper = results_df.pivot(index='model', columns='test_battery', values=['RMSE', 'MAE'])
print('\nTabla de resultados por bateria de test (analoga a la Tabla 3 del paper):')
tabla_tipo_paper.round(3)

## 8. SOH predicho vs. real por bateria de test (cf. Fig. 10-12 del paper)

In [ ]:
fig, axes = plt.subplots(1, 3, figsize=(16, 4.5), sharey=True)
colors = {'Transformer-KAN': 'tab:red', 'KAN': 'tab:green', 'Transformer': 'tab:blue', 'CNN-LSTM': 'tab:purple'}
for ax, test_name in zip(axes, names):
    d = preds_store[test_name]
    cycles = np.arange(L_WINDOW - 1, L_WINDOW - 1 + len(d['true']))
    ax.plot(cycles, d['true'] * 100, 'k--', label='SOH real', linewidth=1.8)
    for m in MODEL_BUILDERS:
        ax.plot(cycles, d[m] * 100, color=colors[m], label=m, alpha=0.8, linewidth=1.2)
    ax.set_title(f'Test = {test_name} (entrenado con las otras dos)')
    ax.set_xlabel('numero de ciclo')
axes[0].set_ylabel('SOH (%)')
axes[0].legend(fontsize=8)
plt.tight_layout(); plt.show()

avg_tabla = results_df.groupby('model')[['RMSE', 'MAE']].mean().sort_values('RMSE')
print('Promedio sobre las 3 baterias de test (analogo al resumen de la Seccion 4.2 / Conclusiones):')
avg_tabla.round(3)

## 9. Comparacion con los resultados reportados en el paper (Tabla 3, Conclusiones)

El paper reporta, sobre el dataset NASA real (promedio de las 3 configuraciones de validacion cruzada, Tabla 3 y Seccion 5 Conclusiones):

| Modelo | RMSE (%) prom. | MAE (%) prom. |
|---|---|---|
| Transformer | 2.17 | 1.85 |
| KAN | 2.28 | 1.92 |
| CNN-LSTM | 1.96 | 1.52 |
| **Transformer-KAN** | **1.58** | **1.33** |

con una mejora de Transformer-KAN de aproximadamente 20% en RMSE y 15% en MAE frente al mejor baseline (CNN-LSTM). A continuacion mostramos, en las mismas unidades, la tabla obtenida en este cuaderno con datos sinteticos.

In [ ]:
paper_avg = pd.DataFrame({
    'Transformer': [2.17, 1.85],
    'KAN': [2.28, 1.92],
    'CNN-LSTM': [1.96, 1.52],
    'Transformer-KAN': [1.58, 1.33],
}, index=['RMSE_paper(%)', 'MAE_paper(%)']).T

comparacion = paper_avg.join(avg_tabla.rename(columns={'RMSE': 'RMSE_propio(%)', 'MAE': 'MAE_propio(%)'}))
print('Paper (NASA real) vs. este cuaderno (baterias sinteticas), promedio de las 3 validaciones cruzadas:')
comparacion.round(3)

### Nota honesta sobre los resultados

Este cuaderno implementa fielmente el **mecanismo central** del paper (la capa KAN con splines-B de Ec. 8-12 y grid/orden de la Tabla 2, el encoder Transformer de la Fig. 7 con los hiperparametros exactos de la Tabla 2, el modelo hibrido completo del Algoritmo 1, la extraccion de las 5 Health Features de la Seccion 2, y el protocolo de validacion cruzada entre baterias de la Seccion 4.2), pero **no es una replica numerica del paper**, por razones declaradas explicitamente:

- **Datos sinteticos:** el dataset NASA real (B0005/B0006/B0007) no esta empaquetado en este entorno. Generamos baterias sinteticas que preservan la estructura fisica del problema (degradacion de capacidad no monotona, curvas CC-CV, pico de temperatura, curva IC), pero no son medidas reales: nuestras Health Features estan mas limpiamente correlacionadas con el SOH sintetico (por construccion) de lo que probablemente lo estan las HF reales con el SOH real del dataset NASA, cuyo ruido de medicion y variabilidad electroquimica son mas dificiles de imitar fielmente que la tendencia general.
- **Escala reducida:** usamos 3 baterias sinteticas de ~165 ciclos cada una (similar en tamano al dataset NASA real), pero un unico `L_WINDOW=10` y un numero de epocas (150) elegido para mantener el tiempo de ejecucion razonable en un cuaderno, sin la busqueda de hiperparametros ni el hardware (RTX 3060) que describe la Seccion 3.4 del paper.
- **Curva IC (HF5):** en vez de derivarla numericamente de una curva de voltaje sintetica (lo que no produciria los picos multiples caracteristicos de una curva $dQ/dV$ real, que provienen de transiciones de fase electroquimicas), la modelamos directamente como una mezcla de gaussianas con amplitud dependiente del SOH, una simplificacion declarada en la Seccion 2 de este cuaderno.
- **Patron de resultados:** en nuestras pruebas de verificacion, Transformer-KAN supera consistentemente al Transformer puro (igual que en el paper), pero el patron completo de la Tabla 3 -Transformer-KAN por encima tambien de KAN puro y CNN-LSTM- **no siempre se reproduce** con estos datos sinteticos: al ser las Health Features casi perfectamente lineales respecto al SOH por construccion (correlaciones >0.98, Seccion 3), un modelo mas simple (KAN puro o CNN-LSTM) puede ajustar la relacion igual de bien o mejor con menos parametros y menos riesgo de sobreajuste con dropout 0.3, mientras que el Transformer completo necesita mas datos/epocas de los que este cuaderno usa para explotar plenamente su capacidad de atencion. Los numeros exactos observados en la celda de resultados (Seccion 8-9) se muestran tal cual, sin forzar una conclusion mas favorable de la que los propios experimentos sostienen; en el dataset NASA real, con series mas largas, mas ruido y dependencias temporales genuinamente no lineales, es plausible que la ventaja de Transformer-KAN reportada por los autores sea mas robusta que en este entorno sintetico simplificado.